# How to Fine-Tune a Decision Model with TRL (SFT + LoRA)


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Liquid4All/cookbook/blob/main/finetuning/notebooks/sft_for_decision_models_with_trl.ipynb)

This tutorial shows you how to fine-tune a decision model, a model that makes decisions instead of generating text, on your own labels.

We fine-tune our [`d1-3B`](https://huggingface.co/LiquidAI/d1-3B) decision model with Supervised Fine-tuning (SFT) and LoRA using the TRL library, with intent classification for customer support routing on Banking77 as the example.

Follow along if it's your first time using trl, or take single code snippets for your own workflow

## 🎯 What You'll Find:
- **LoRA + SFT** - Fine-tune the d1-3B decision model on your own decisions using LoRA (from PEFT)
- **Evaluation** - Compare the decision model's accuracy before and after fine-tuning

## 📋 Prerequisites:
- **GPU Runtime**: Select a GPU in `Runtime` → `Change runtime type`. A free T4 GPU is enough (about 35 minutes of training)
- **Hugging Face Account**: Only needed to push your model to the Hub

# 📦 Installation & Setup

First, let's install all the required packages:

In [1]:
!pip install -qqq "transformers==5.19.0" "trl==1.15.0" "peft==0.21.2" "torchao>0.16.0" datasets --progress-bar off

Let's now verify the packages are installed correctly

In [2]:
import torch
import transformers
import trl
import peft
import os
os.environ["WANDB_DISABLED"] = "true"

print(f"📦 PyTorch version: {torch.__version__}")
print(f"🤗 Transformers version: {transformers.__version__}")
print(f"📊 TRL version: {trl.__version__}")
print(f"🎛️ PEFT version: {peft.__version__}")

# bfloat16 on GPUs that support it natively (e.g. A100), float16 otherwise (e.g. T4)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16
print(f"🔢 Precision: {dtype}")

📦 PyTorch version: 2.11.0+cu130
🤗 Transformers version: 5.19.0
📊 TRL version: 1.15.0
🎛️ PEFT version: 0.21.2
🔢 Precision: torch.float16


# Loading the model from Transformers 🤗

d1-3B ships its own code for the System One API (`system_one` and `system_one_batch`), so we load it with `trust_remote_code=True`.

We also download the model repository locally. It contains `prompt.py`, the module d1-3B uses to turn a state and a question into a prompt. We will reuse it to build our training data, so that the model sees the exact same prompt during training and inference.

In [3]:
import sys
from huggingface_hub import snapshot_download
from transformers import AutoModel, AutoTokenizer

model_id = "LiquidAI/d1-3B"

# Download the checkpoint and d1's own code, and make its prompt builder importable
local_dir = snapshot_download(model_id)
sys.path.insert(0, local_dir)

print("📚 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("🧠 Loading model...")
model = AutoModel.from_pretrained(
    model_id,
    trust_remote_code=True,
    dtype=dtype,
    ).to("cuda")

print("✅ Model loaded successfully!")
print(f"🔢 Parameters: {model.num_parameters():,}")
print(f"📖 Vocab size: {len(tokenizer)}")
print(f"💾 Model size: ~{model.num_parameters() * 2 / 1e9:.1f} GB ({dtype})")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

📚 Loading tokenizer...
🧠 Loading model...


[transformers] A new version of the following files was downloaded from https://huggingface.co/LiquidAI/d1-3B:
- hybrid.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] A new version of the following files was downloaded from https://huggingface.co/LiquidAI/d1-3B:
- prompt.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] A new version of the following files was downloaded from https://huggingface.co/LiquidAI/d1-3B:
- api.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] A new version of the following files was downloaded from https://huggingface.co/LiquidAI/d1-3B:
- lfm2_vl.py
- runner.py
. Make sure to double-check they do not contain 

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

✅ Model loaded successfully!
🔢 Parameters: 3,123,483,888
📖 Vocab size: 125017
💾 Model size: ~6.2 GB (torch.float16)


# 💡 What is a decision model?

A decision model is a model that makes decisions instead of generating text: it answers yes/no, multiple-choice, and scoring questions with a probability for each answer, in a single forward pass. Typical uses are classification, routing and triage, content moderation, and LLM-as-a-judge scoring.

d1-3B is a 3B parameter decision model built on [LFM2.5-VL-3B](https://huggingface.co/LiquidAI/LFM2.5-VL-3B). It takes a state and a set of questions, and returns typed answers in one forward pass, without generating any output tokens. Each question uses one of the following three primitives:

- `noul`: Yes or no, as P(yes)
- `choice`: One of named options, with a probability for each
- `score`: A level on an ordered scale of 2 to 10 levels

Before we fine-tune, let's see how d1-3B makes a decision. We ask a `choice` question about a customer message:

In [4]:
state = "I was charged twice this month, please refund one of them."

questions = {
    "team": {
        "type": "choice",
        "instructions": "Which team should handle this?",
        "criteria": {
            "billing": "Charges, refunds, invoices",
            "technical": "App or site faults",
            "fraud": "Suspected unauthorised use",
        },
    },
}

response = model.system_one(state=state, questions=questions)

print(response)

{'answers': {'team': {'type': 'choice', 'choice': 'billing', 'confidence': 0.9835613206479191, 'probabilities': {'billing': 0.9835613206479191, 'technical': 0.01059020995608956, 'fraud': 0.005848469395991271}}}, 'usage': {'input_tokens': 61, 'output_tokens': 0}}


Under the hood, d1-3B renders the state and the question into a single prompt. Each option gets a short code (`A`, `B`, `C`, ...), and the prompt ends right after the assistant header:

In [5]:
from prompt import aliases, as_question, render

rendered_prompt = render(
    tokenizer=tokenizer,
    state=state,
    q=as_question(questions["team"]),
    bos=tokenizer.bos_token,
)

print(rendered_prompt)

<|startoftext|><|im_start|>user
I was charged twice this month, please refund one of them.


QUESTION:
Which team should handle this?

Options:
A Charges, refunds, invoices
B App or site faults
C Suspected unauthorised use

Reply with the option code only.<|im_end|>
<|im_start|>assistant



The model then runs one forward pass and reads the next-token probabilities at the end of the prompt, the **answer slot**. It only looks at the tokens of the option codes and normalizes them into the probabilities you saw above.

This means that fine-tuning a decision model is regular SFT where **the completion is a single token: the code of the correct option**, followed by the end-of-turn token `<|im_end|>`.

# 🎯 Supervised Fine-Tuning (SFT + LoRA)

SFT teaches the model to make your decisions by training on input-output pairs. For a decision model, the input is the rendered prompt and the output is the code of the correct option.

We use LoRA (Low-Rank Adaptation) to finetune the model by only training a small number of additional parameters.

## Load an SFT Dataset

We will use [Banking77](https://huggingface.co/datasets/mteb/banking77), a dataset of ~13k online banking customer messages, each labeled with one of 77 fine-grained intents. To keep this notebook fast on a free T4 GPU, we train on 3,000 messages and evaluate on 100. Feel free to increase these numbers with `select(range(...))`.

In [6]:
from datasets import load_dataset

print("📥 Loading dataset...")
dataset = load_dataset("mteb/banking77")
train_dataset = dataset["train"].shuffle(seed=42).select(range(3000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(100))

print("✅ Dataset loaded:")
print(f"   📚 Train samples: {len(train_dataset)}")
print(f"   🧪 Test samples: {len(test_dataset)}")
print(f"\n📝 Single Sample: {train_dataset[0]}")

📥 Loading dataset...


README.md:   0%|          | 0.00/16.6k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  294kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 91.4kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9993 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3076 [00:00<?, ? examples/s]

✅ Dataset loaded:
   📚 Train samples: 3000
   🧪 Test samples: 100

📝 Single Sample: {'text': 'How long is a top up supposed to be pending, because an hour is too long! Can you fix this to go through?', 'label': 47, 'label_text': 'pending_top_up'}


## Define the decision

We turn the classification task into a single `choice` question. Each intent becomes one option, described by its label name.

**Important**: The option codes are assigned by position. Use the same question, with the options in the same order, for training and inference.

In [7]:
labels = sorted(set(dataset["train"]["label_text"]))

intent_question = {
    "type": "choice",
    "instructions": "Which intent does this banking customer message express?",
    "criteria": {label: label.replace("_", " ") for label in labels},
}

# The code d1-3B assigns to each option, e.g. "card_arrival" -> "12"
codes = {label: code for label, (code, _) in zip(labels, aliases(tokenizer, labels))}

print(f"🏷️ Number of options: {len(labels)}")
print(f"🔤 First option codes: {dict(list(codes.items())[:5])}")

🏷️ Number of options: 77
🔤 First option codes: {'Refund_not_showing_up': '00', 'activate_my_card': '01', 'age_limit': '02', 'apple_pay_or_google_pay': '03', 'atm_support': '04'}


With up to 26 options, d1-3B uses the letters `A` to `Z` as codes. With more options, as here, it switches to the two-digit codes `00` to `76`, each of which is a single token.

We now transform the dataset to a format compatible with `trl`. We use TRL's **prompt-completion** format:
- `prompt`: the full prompt rendered by d1-3B's own prompt builder
- `completion`: the code of the correct intent, followed by `<|im_end|>`

With a prompt-completion dataset, `SFTTrainer` computes the loss only on the completion, so the model learns to put its probability mass on the right option code.

In [8]:
question = as_question(intent_question)

def to_prompt_completion(example):
    return {
        "prompt": render(tokenizer, example["text"], question, bos=tokenizer.bos_token),
        "completion": codes[example["label_text"]] + tokenizer.eos_token,
    }

split = train_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset_sft = split["train"].map(to_prompt_completion, remove_columns=split["train"].column_names)
eval_dataset_sft = split["test"].map(to_prompt_completion, remove_columns=split["test"].column_names)

print("✅ SFT Dataset formatted:")
print(f"   📚 Train samples: {len(train_dataset_sft)}")
print(f"   🧪 Eval samples: {len(eval_dataset_sft)}")
print(f"\n📝 Prompt:\n{train_dataset_sft[0]['prompt']}")
print(f"\n🎯 Completion: {train_dataset_sft[0]['completion']!r}")

Map:   0%|          | 0/2700 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

✅ SFT Dataset formatted:
   📚 Train samples: 2700
   🧪 Eval samples: 300

📝 Prompt:
<|startoftext|><|im_start|>user
So, I was under the impression that you have an offer for crypto top up. This doesn't seem to be working. Now, the money has just disappeared from my account. What is the deal?


QUESTION:
Which intent does this banking customer message express?

Options:
00 Refund not showing up
01 activate my card
02 age limit
03 apple pay or google pay
04 atm support
05 automatic top up
06 balance not updated after bank transfer
07 balance not updated after cheque or cash deposit
08 beneficiary not allowed
09 cancel transfer
10 card about to expire
11 card acceptance
12 card arrival
13 card delivery estimate
14 card linking
15 card not working
16 card payment fee charged
17 card payment not recognised
18 card payment wrong exchange rate
19 card swallowed
20 cash withdrawal charge
21 cash withdrawal not recognised
22 change pin
23 compromised card
24 contactless not working
25 country s

## Evaluate the base model

Let's measure how well d1-3B performs on the 100 test messages before fine-tuning. `system_one_batch` packs many messages into each forward pass.

In [9]:
from tqdm.auto import tqdm

def evaluate(model, dataset, batch_size=16):
    """Accuracy of the model's intent decisions on `dataset`, and its predictions."""
    predictions = []
    # Evaluate in batches, so the progress bar shows how far we are
    for start in tqdm(range(0, len(dataset), batch_size), desc="Evaluating"):
        texts = dataset[start:start + batch_size]["text"]
        responses = model.system_one_batch([(text, {"intent": intent_question}) for text in texts])
        predictions += [response["answers"]["intent"]["choice"] for response in responses]
    accuracy = sum(p == y for p, y in zip(predictions, dataset["label_text"])) / len(dataset)
    return accuracy, predictions

print("📏 Evaluating the base model...")
baseline_accuracy, baseline_predictions = evaluate(model, test_dataset)
print(f"🎯 Baseline accuracy: {baseline_accuracy:.1%}")

📏 Evaluating the base model...


Evaluating:   0%|          | 0/7 [00:00<?, ?it/s]

🎯 Baseline accuracy: 83.0%


Let's free up the GPU memory before training:

In [10]:
import gc

del model
gc.collect()
torch.cuda.empty_cache()

## Load the model for training

For training, we load the same weights with the standard Transformers class. d1-3B's checkpoint uses the standard LFM2-VL module names, so we don't need its custom code here.

In [11]:
from transformers import AutoModelForImageTextToText

print("🧠 Loading model for training...")
model = AutoModelForImageTextToText.from_pretrained(model_id, dtype=dtype, device_map="auto")
print("✅ Model loaded successfully!")

🧠 Loading model for training...


Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

✅ Model loaded successfully!


## Wrap the model with PEFT

We specify target modules that will be finetuned while the rest of the models weights remains frozen. Feel free to modify the `r` (rank) value:
- Higher → better approximation of full-finetuning
- Lower → needs even less compute resources

Our task is text-only, so we only adapt the language model: its attention layers (`q_proj`, `k_proj`, `v_proj`, `out_proj`), its short convolution layers (`in_proj`, `out_proj`), and its MLP layers (`w1`, `w2`, `w3`). The vision encoder uses some of the same layer names, so we exclude it explicitly.

In [12]:
from peft import LoraConfig, get_peft_model

target_modules = [
    "q_proj", "k_proj", "v_proj",
    "out_proj", "in_proj",
    "w1", "w2", "w3"
]

peft_config = LoraConfig(
    r=16,  # <- lower values = fewer parameters
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
    exclude_modules=r".*vision_tower.*", # The vision encoder also has q_proj, k_proj, v_proj, and out_proj layers: keep them frozen by excluding them
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

print("✅ LoRA configuration created!")
print(f"🎛️  LoRA rank: {peft_config.r}")
print(f"📊 LoRA alpha: {peft_config.lora_alpha}")

trainable params: 24,461,312 || all params: 3,147,945,200 || trainable%: 0.7771
✅ LoRA configuration created!
🎛️  LoRA rank: 16
📊 LoRA alpha: 32


## Launch Training

We are now ready to launch an SFT run with `SFTTrainer`, feel free to modify `SFTConfig` to play around with different configurations.

In [13]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir="./d1-3B-banking77",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # <- effective batch size of 16
    gradient_checkpointing=True,  # <- trades compute for memory, so it fits on a T4
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.05,  # <- a float is a fraction of the total steps
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="no",
    bf16=dtype == torch.bfloat16,
    fp16=dtype == torch.float16,
    report_to="none",
)

print("🏗️  Creating SFT trainer...")
sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset_sft,
    eval_dataset=eval_dataset_sft,
    processing_class=tokenizer,
)

print("\n🚀 Starting SFT training...")
sft_trainer.train()

print("🎉 SFT training completed!")

sft_trainer.save_model()
print(f"💾 Saving to: {sft_config.output_dir}")

🏗️  Creating SFT trainer...


Adding EOS to train dataset:   0%|          | 0/2700 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2700 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2700 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2700 [00:00<?, ? examples/s]

...

Dropping fully masked examples from eval dataset:   0%|          | 0/300 [00:00<?, ? examples/s]


🚀 Starting SFT training...


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
50,0.252172,0.277826,0.279177,0.917763,372382.000000
100,0.199836,0.226370,0.244880,0.937500,745028.000000
150,0.205320,0.210120,0.205339,0.937500,1117273.000000
169,0.215371,0.208865,0.206764,0.937500,1257044.000000


🎉 SFT training completed!
💾 Saving to: ./d1-3B-banking77


## Save merged model

Merge the extra weights learned with LoRA back into the model to obtain a "normal" model checkpoint.

The merged checkpoint contains only the weights and the tokenizer. To keep the System One API, we also copy d1-3B's own code and processor configuration into the same folder.

In [14]:
import shutil

output_dir = "./d1-3B-banking77-merged"

print("🔄 Merging LoRA weights...")
merged_model = sft_trainer.model.merge_and_unload()
merged_model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

# d1-3B's code for the System One API, its processor configuration, and its license
for filename in ["api.py", "hybrid.py", "lfm2_vl.py", "modeling_d1.py", "prompt.py", "runner.py", "processor_config.json", "LICENSE"]:
    shutil.copy(os.path.join(local_dir, filename), output_dir)

print(f"💾 Merged model saved to: {output_dir}")

# To push the merged model and d1-3B's code to the Hugging Face Hub, uncomment:
# from huggingface_hub import create_repo, upload_folder
# create_repo("your-model-path", exist_ok=True)
# upload_folder(folder_path=output_dir, repo_id="your-model-path")

🔄 Merging LoRA weights...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾 Merged model saved to: ./d1-3B-banking77-merged


# 📊 Evaluate the Fine-tuned Model

Let's free up the GPU memory again and load the fine-tuned model the same way we loaded d1-3B: with `AutoModel` and `trust_remote_code=True`.

In [15]:
del model, merged_model, sft_trainer
gc.collect()
torch.cuda.empty_cache()

finetuned = AutoModel.from_pretrained(output_dir, trust_remote_code=True, dtype=dtype).to("cuda")

print("📏 Evaluating the fine-tuned model...")
finetuned_accuracy, finetuned_predictions = evaluate(finetuned, test_dataset)
print(f"🎯 Baseline accuracy:   {baseline_accuracy:.1%}")
print(f"🎯 Fine-tuned accuracy: {finetuned_accuracy:.1%}")

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

📏 Evaluating the fine-tuned model...


Evaluating:   0%|          | 0/7 [00:00<?, ?it/s]

🎯 Baseline accuracy:   83.0%
🎯 Fine-tuned accuracy: 91.0%
